In [1]:
import vitaldb
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
vf = vitaldb.VitalFile("ICU.vital") #load
print(vf.get_track_names()) #check

['Intellivue/ABP_MEAN', 'Intellivue/ABP_SYS', 'Intellivue/ABP_DIA', 'Intellivue/ABP_HR', 'Intellivue/CVP_MEAN', 'Intellivue/PLETH_SAT_O2', 'Intellivue/PLETH_HR', 'Intellivue/PLETH_PERF_REL', 'Intellivue/NIBP_SYS', 'Intellivue/NIBP_DIA', 'Intellivue/NIBP_MEAN', 'Intellivue/NIBP_HR', 'Intellivue/ECG_HR', 'Intellivue/RR', 'Intellivue/ST_II', 'Intellivue/ECG_VPC_CNT', 'Intellivue/QT_GL', 'Intellivue/QTc', 'Intellivue/QTc_DELTA', 'Intellivue/QT_HR', 'Intellivue/ABP', 'Intellivue/CVP', 'Intellivue/ECG_II', 'Intellivue/PLETH']


In [3]:
#Fs check
for track_name, track in vf.trks.items():
    print(f"{track_name:40s} | {track.srate} Hz")

Intellivue/ABP_MEAN                      | 1.0 Hz
Intellivue/ABP_SYS                       | 1.0 Hz
Intellivue/ABP_DIA                       | 1.0 Hz
Intellivue/ABP_HR                        | 1.0 Hz
Intellivue/CVP_MEAN                      | 1.0 Hz
Intellivue/PLETH_SAT_O2                  | 1.0 Hz
Intellivue/PLETH_HR                      | 1.0 Hz
Intellivue/PLETH_PERF_REL                | 1.0 Hz
Intellivue/NIBP_SYS                      | 1.0 Hz
Intellivue/NIBP_DIA                      | 1.0 Hz
Intellivue/NIBP_MEAN                     | 1.0 Hz
Intellivue/NIBP_HR                       | 1.0 Hz
Intellivue/ECG_HR                        | 1.0 Hz
Intellivue/RR                            | 1.0 Hz
Intellivue/ST_II                         | 1.0 Hz
Intellivue/ECG_VPC_CNT                   | 1.0 Hz
Intellivue/QT_GL                         | 1.0 Hz
Intellivue/QTc                           | 1.0 Hz
Intellivue/QTc_DELTA                     | 1.0 Hz
Intellivue/QT_HR                         | 1.0 Hz


In [ ]:
# Extract by sample plate
ecg_data    = vf.to_numpy(['Intellivue/ECG_II'], 1/500)
wave125     = vf.to_numpy(['Intellivue/ABP', 'Intellivue/CVP', 'Intellivue/PLETH'], 1/125)
numeric_tracks = [
    'Intellivue/ABP_SYS', 'Intellivue/ABP_MEAN', 'Intellivue/ABP_DIA',
    'Intellivue/ABP_HR', 'Intellivue/CVP_MEAN',
    'Intellivue/PLETH_SAT_O2', 'Intellivue/PLETH_HR',
    'Intellivue/NIBP_SYS', 'Intellivue/NIBP_MEAN', 'Intellivue/NIBP_DIA',
    'Intellivue/ECG_HR', 'Intellivue/RR', 'Intellivue/ST_II',
]
num_data = vf.to_numpy(numeric_tracks, 1.0)

t_ecg = np.arange(ecg_data.shape[0]) / 500
t_125 = np.arange(wave125.shape[0]) / 125
t_num = np.arange(num_data.shape[0])

# ── Figure 1: Waveforms ───────────────────────────────────────
wave_info = [
    ('ECG II', t_ecg, ecg_data[:, 0]),
    ('ABP',    t_125, wave125[:, 0]),
    ('CVP',    t_125, wave125[:, 1]),
    ('PLETH',  t_125, wave125[:, 2]),
]
fig1 = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=[w[0] for w in wave_info],
    vertical_spacing=0.05
)

for i, (name, t, sig) in enumerate(wave_info, start=1):
    fig1.add_trace(
        go.Scattergl(x=t, y=sig, mode='lines', name=name,
                     line=dict(width=0.8), connectgaps=False),
        row=i, col=1
    )

fig1.update_layout(
    height=900, title="Waveform Signals (500Hz / 125Hz)",
    hovermode='x unified', dragmode='zoom',
)
fig1.update_xaxes(zeroline=False)
fig1.update_yaxes(zeroline=False)
fig1.update_xaxes(title_text="Time (s)", row=4, col=1)
fig1.write_html("waveform.html")

In [ ]:
# ── Figure 2: Numeric Parameters (1Hz) ───────────────────────
groups = {
    'BP (mmHg)':  ['Intellivue/ABP_SYS', 'Intellivue/ABP_MEAN', 'Intellivue/ABP_DIA',
                   'Intellivue/NIBP_SYS', 'Intellivue/NIBP_MEAN', 'Intellivue/NIBP_DIA'],
    'HR (bpm)':   ['Intellivue/ABP_HR', 'Intellivue/PLETH_HR', 'Intellivue/ECG_HR'],
    'SpO2 / CVP': ['Intellivue/PLETH_SAT_O2', 'Intellivue/CVP_MEAN'],
    'RR / ST_II': ['Intellivue/RR', 'Intellivue/ST_II'],
}

fig2 = make_subplots(
    rows=len(groups), cols=1,
    shared_xaxes=True,
    subplot_titles=list(groups.keys()),
    vertical_spacing=0.06
)

for row_i, (group_name, tracks) in enumerate(groups.items(), start=1):
    for track in tracks:
        if track in numeric_tracks:
            idx = numeric_tracks.index(track)
            fig2.add_trace(
                go.Scatter(x=t_num, y=num_data[:, idx],
                           mode='lines', name=track.split('/')[-1],
                           connectgaps=False),
                row=row_i, col=1
            )

fig2.update_layout(
    height=800, title="Numeric Parameters (1Hz)",
    hovermode='x unified', dragmode='zoom',
)
fig2.update_xaxes(zeroline=False)
fig2.update_yaxes(zeroline=False)
fig2.update_xaxes(title_text="Time (s)", row=len(groups), col=1)
fig2.write_html("numeric.html")